In [ ]:
import pandas as pd
import re
import json
import urllib
from sqlalchemy import create_engine

# --- Load SQL credentials from JSON ---
with open("credentials_resv_data_access_SQL_20250528.json", "r") as f:
    config = json.load(f)

sql_config = config["sql_server_config"]
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={sql_config['server']};"
    f"DATABASE={sql_config['database']};"
    f"UID={sql_config['username']};"
    f"PWD={sql_config['password']}"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# --- Load SQL query from external file ---
def query_station_metadata():
    with open("entire_reservior_data_set_with_metadata_inactive_and_active_obsv_and_manual_20250528.sql", "r") as file:
        sql_query = file.read()
    return pd.read_sql(sql_query, engine)

# --- Extract Reservoir Name (handles releases/outflows too) ---
def extract_base_name(name):
    name = str(name).upper()
    name = re.sub(r'^\d+\s*', '', name)
    name = re.sub(r'\b(USBR|USGS|RELEASE|OUTFLOW|ELEVATION|STORAGE|CONTENTS?|POOL|EVAPORATION|LEVEL|OUTLET|OBSERVATION|SITE|RES\.?|RSVR\.?)\b', '', name)
    name = re.sub(r'[^A-Z0-9 ]+', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    match = re.search(r'((?:[A-Z0-9]+\s){0,3})RESERVOIR', name)
    return f"{match.group(1).strip()} Reservoir".title() if match else name.title()

# --- Determine if a station is a release/outflow ---
def is_outflow(name):
    name = str(name).upper()
    return bool(re.search(r'RELEASE|OUTFLOW', name))

# --- Final Export Function ---
def generate_final_export(df):
    df = df[df["SiteType"].isin(["Reservoir", "Reservoir Release"])]
    df["NewSiteName"] = df["MasterStationName"].apply(extract_base_name)
    df["IsOutflow"] = df["MasterStationName"].apply(is_outflow)

    grouped_rows = []
    counter = 1

    for site in df["NewSiteName"].unique():
        site_df = df[df["NewSiteName"] == site]

        main_group = site_df[~site_df["IsOutflow"]].copy()
        outflow_group = site_df[site_df["IsOutflow"]].copy()

        for group_df in [main_group, outflow_group]:
            if group_df.empty:
                continue
            site_id = f"PD{str(counter).zfill(3)}"
            counter += 1
            group_df = group_df.reset_index(drop=True)
            group_df["RecordNum"] = group_df.index + 1

            transposed = group_df[["NewSiteName"]].drop_duplicates()
            transposed["SiteID (New)"] = site_id

            # Add StationPage URL
            station_ids = group_df["MasterStationID"].dropna().astype(str).tolist()
            station_id_str = ",".join(station_ids)
            station_page_url = f"https://waterrights.utah.gov/dvrtdb/daily-chart.asp?STATION_ID={station_id_str}"
            transposed["StationPage"] = station_page_url

            for col in ["MasterStationName", "MasterStationID", "UNITS_DESC_ENTRY", "CollectionStationName", "COLLECTION_SYSTEM"]:
                pivot = group_df.pivot_table(index="NewSiteName", columns="RecordNum", values=col, aggfunc="first")
                pivot.columns = [f"{col} {i}" for i in pivot.columns]
                transposed = transposed.merge(pivot, on="NewSiteName", how="left")

            grouped_rows.append(transposed)

    final_df = pd.concat(grouped_rows, ignore_index=True)
    final_df = final_df.rename(columns={
        "MasterStationID": "Station_ID (old)",
        "COLLECTION_SYSTEM": "CollectionSystemName"
    })
    final_df["Check"] = ""
    final_df["Comment"] = ""
    final_df.to_csv("Reservior_Station_Naming_Metadata_Transposed_20250530.csv", index=False)
    print("✅ Saved: Reservior_Station_Naming_Metadata_Transposed_20250530.csv")

# --- MAIN ---
def main():
    print("🔄 Loading station metadata...")
    df = query_station_metadata()
    print(f"📦 Retrieved {len(df)} rows")
    generate_final_export(df)
    print("✅ Done.")

if __name__ == "__main__":
    main()